In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from collections import Counter
from imblearn.over_sampling import SMOTE

In [5]:
df=pd.read_csv("../data/CleanedSupplyChainDataset.csv", encoding='latin-1')

In [7]:
# Convert to datetime if needed
df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

# Create time-based features
df["order_month"] = df["order date (DateOrders)"].dt.month
df["order_day"] = df["order date (DateOrders)"].dt.day_name()
df["order_hour"] = df["order date (DateOrders)"].dt.hour

In [8]:
features = [
    "Type",
    "Days for shipment (scheduled)",
    "Sales per customer",
    "Sales",
    "Product Price",
    "Category Name",
    "Customer Segment",
    "Department Name",
    "Customer Country",
    "Order Region",
    "Shipping Mode",
    "Order Item Profit Ratio",
    "order_month",
    "order_day",      # if engineered
    "order_hour"
]

X = df[features]
y = df["Late_delivery_risk"]

In [9]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns:', cat_cols)

# Frequency encoding (low-dimensional and robust for high-cardinality)
for col in cat_cols:
    freq = X[col].value_counts(normalize=True)
    X[f'{col}_freq'] = X[col].map(freq)

# Keep numeric columns + new encoded features, drop original string categories
X_encoded = X.drop(columns=cat_cols)
print('Shape after freq+target encoding:', X_encoded.shape)

# use encoded features for modeling
X = X_encoded

# train/test split after encoding
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Categorical columns: ['Type', 'Category Name', 'Customer Segment', 'Department Name', 'Customer Country', 'Order Region', 'Shipping Mode', 'order_day']
Shape after freq+target encoding: (172765, 15)


C:\Users\SOHAM\AppData\Local\Temp\ipykernel_83924\1639087433.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[f'{col}_freq'] = X[col].map(freq)
C:\Users\SOHAM\AppData\Local\Temp\ipykernel_83924\1639087433.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[f'{col}_freq'] = X[col].map(freq)
C:\Users\SOHAM\AppData\Local\Temp\ipykernel_83924\1639087433.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

In [10]:
# Balancing the training data using SMOTE
print("Before balancing (train):", Counter(y_train))

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After balancing (train):", Counter(y_train_bal))

Before balancing (train): Counter({1: 79182, 0: 59030})
After balancing (train): Counter({0: 79182, 1: 79182})


In [11]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"\n--- {model_name} ---")
    print("Accuracy:", round(accuracy_score(y_true, y_pred),2))
    print("Precision:", round(precision_score(y_true, y_pred),2))
    print("Recall:", round(recall_score(y_true, y_pred),2))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

In [12]:
# Fit the Random Forest model on balanced data
rf_model_balanced = RandomForestClassifier(random_state=42)
rf_model_balanced.fit(X_train_bal, y_train_bal)

y_pred_rf_balanced = rf_model_balanced.predict(X_test)

evaluate_model(y_test, y_pred_rf_balanced, "Random Forest Classifier")


--- Random Forest Classifier ---
Accuracy: 0.75
Precision: 0.84
Recall: 0.7

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.82      0.74     14758
           1       0.84      0.70      0.76     19795

    accuracy                           0.75     34553
   macro avg       0.75      0.76      0.75     34553
weighted avg       0.77      0.75      0.75     34553

